# Demo: Building a RAG-powered FAQ Agent with Custom Knowledge

# Step 1: Install required packages

In [ ]:
%pip install -U langchain langchain-openai langchain-community langchain-classic faiss-cpu tiktoken python-dotenv

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------------------------------- -- 2.4/2.5 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 11.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 25.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ------- -------------------------------- 3.4/18.9 MB 15.5 MB/s eta 0:00:01
   ---------------- ----------------------- 7.9/18.9 MB 18.7 MB/s eta 0:00:01
   ---------------------------- ----------- 13.6/18.9 MB 21.9 MB/s eta 0:00:01
   ---------------------------------------  18.9/18.9 MB 23.4 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 22.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/878.7 kB ? eta -:--:--
   ----------------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\AgrawalN\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Step 2: Import dependencies

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

# Step 3: Set Azure OpenAI credentials



In [ ]:
# Load environment variables from .env file (never hardcode secrets in notebooks)
load_dotenv()

# Verify the key was loaded
if not os.environ.get("AZURE_OPENAI_API_KEY"):
    raise ValueError("AZURE_OPENAI_API_KEY not found. Make sure it is set in your .env file.")

# Step 4: Load and chunk your custom FAQ document


In [4]:
# Load the FAQ document and split it into chunks for embedding

# loading the txt  document file
# langchain supports many types of document formats, such as PDF, Word, HTML, etc. 
# You can use the appropriate loader for your document type. In this example, we are using TextLoader for a plain text file.
loader = TextLoader("faq.txt")  # Ensure this file exists
documents = loader.load()

# look in text for by default \n\n characters paragraph breaks. If it doesn't find the character, it will take the first 500 characters as a chunk
# chunk overlap means a certain characters can be in multiple chunks.
# if we want to change the separator, we can use the separator parameter in the CharacterTextSplitter.
# For example, if we want to split by sentences, we can use separator="\n\n" or separator="," depending on how the document is structured.
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

In [5]:
docs

[Document(metadata={'source': 'faq.txt'}, page_content='COMPANY FAQ - TechStore Customer Support\n\n========================================\nSHIPPING & DELIVERY\n========================================'),
 Document(metadata={'source': 'faq.txt'}, page_content='Q: What is your shipping policy?\nA: We offer multiple shipping options to meet your needs. Standard shipping takes 5-7 business days and costs $5.99. Express shipping takes 2-3 business days and costs $12.99. Premium overnight shipping is available for $24.99. Free standard shipping is available on all orders over $50. All orders are processed within 1-2 business days.'),
 Document(metadata={'source': 'faq.txt'}, page_content='Q: Do you ship internationally?\nA: Yes, we ship to over 150 countries worldwide. International shipping typically takes 10-21 business days depending on the destination. Shipping costs are calculated at checkout based on your location and order weight. Please note that customers are responsible for any 

# Step 5: Create vectorstore using Azure embeddings


In [6]:
# Embed document chunks and store them in a FAISS vector index

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2023-05-15",
    deployment="text-embedding-ada-002", # this is the embedding model used to build the vector store.
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
)
# This might fail if the API key is expired or the model is unavailable.
#FAISS.from_documents()  will embed and generate the vectors and store them
vectorstore = FAISS.from_documents(docs, embeddings)

APIConnectionError: Connection error.

# Step 6: Initialize the Azure OpenAI LLM

In [8]:
# Initialize the GPT-4o model from Azure with temperature 0 - controls the sampling, most likely word.
# it's a setting on a model which determines the probability for deterministic output

llm = AzureChatOpenAI(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2025-01-01-preview",
    deployment_name="gpt-5-mini",
    temperature=0 #conservative, only sample most likely result.
)
#update

# Step 7: Create the RAG chain

In [9]:
# Create a RetrievalQA chain that uses the retriever and LLM to answer queries with source context
# Here is our RAG Pipeline, connecting the vector store to the LLM.

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(), #makes our 
    return_source_documents=True
)

NameError: name 'vectorstore' is not defined

# Step 8: Ask a question

In [ ]:
# Send a question to the RAG chain and store the result

query = "What is our return policy?"
# Typically we would do this in the BE.
# We will have a FE to grab the prompt
result = qa_chain.invoke({"query": query})

# Step 9: Print results

In [ ]:
# Display the final answer and the source chunks used

print("Answer:", result["result"])
print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)